# Styling DataFrames

## Loading Libraries

In [ ]:
# Numerical Computing
import numpy as np

# Data Manipualtion
import pandas as pd

# Data Visualization
import seaborn as sns
import matplotlib.pyplot as plt

# PyArrow
import pyarrow as pa

# SparkLines
import sparklines

### Loading Data

In [ ]:
url = 'https://github.com/mattharrison/datasets/raw/master'\
    '/data/dirtydevil.txt'

df = pd.read_csv(url, skiprows=lambda num: num <34 or num == 35,
                 sep='\t')

In [ ]:
def to_denver_time(df_, time_col, tz_col):
    return (
        df_
        .assign(**{
            tz_col: df_[tz_col].replace('MDT', 'MST7MDT')
        })
        .groupby(tz_col)[time_col]
        .transform(
            lambda s: pd.to_datetime(s)
            .dt.tz_localize(s.name, ambiguous=True)
            .dt.tz_convert('America/Denver')
        )
    )

In [ ]:
def tweak_river(df_):
    return (
        df_
        .assign(
            datetime=to_denver_time(df_, 'datetime', 'tz_cd')
        )
        .rename(columns={
            '144166_00060': 'cfs',
            '144167_00065': 'gage_height'
        })
        .loc[:, [
            'datetime',
            'agency_cd',
            'site_no',
            'tz_cd',
            'cfs',
            'gage_height'
        ]]
        .set_index('datetime')
    )

In [ ]:
dd = tweak_river(df)

In [ ]:
print(dd)

In [ ]:
import sparklines

agg_flow = (
    dd
    # .resample('M')  # resample .agg doesn't support named aggregations
    .groupby(pd.Grouper(freq='M'))
    .agg(
        cfs=('cfs', 'median'),

        total_flow=('cfs',
                    lambda ser: (ser * 15 * 60).sum()),

        gage_height=('gage_height', 'median'),

        flow_trend=('cfs',
                    lambda ser: sparklines.sparklines(
                        ser
                        .fillna(0)
                        .resample('2D')
                        .median()
                        .fillna(0)
                    )[0])
    )

    .assign(
        quarterly_flow=lambda df_: (
            df_
            .total_flow
            .resample('Q')
            .transform('sum')
        ),

        percent_quarterly_flow=lambda df2_: (
            df2_.total_flow / df2_.quarterly_flow
        ),

        off_goal=lambda df3_: (
            df3_.percent_quarterly_flow - .33
        ),

        cost=lambda df4_: (
            df4_.total_flow * .0002
        )
    )
)

### Sparklines

In [ ]:
print(agg_flow.iloc[:, :4])

In [ ]:
sparklines.sparklines(range(10))

In [ ]:
flow_trend = (
    'cfs',
    lambda ser: sparklines.sparklines(
        ser
        .resample('2D')
        .median()
        .fillna(0)
    )[0]
)

In [ ]:
agg_flow.flow_trend

### Formatting

In [ ]:
(
    agg_flow
    .reset_index()
    .style
    .format(
        {
            'cost': '${:,.0f}',
            'datetime': '{:%Y/%m}/01',
            'percent_quarterly_flow': '{:.1%}',
            'off_goal': '{:.1%}',
            **{
                col: '{:.1f}'
                for col in ['cfs', 'total_flow', 'quarterly_flow']
            }
        },
        na_rep='Missing'
    )
)

### Embedding Bar Plots

In [ ]:
(
    agg_flow
    .fillna(0)
    .reset_index()
    .style
    .format(
        {
            'cost': '${:,.0f}',
            'datetime': '{:%Y/%m}/01',
            'percent_quarterly_flow': '{:.1%}',
            'off_goal': '{:.1%}',
            **{
                col: '{:.1f}'
                for col in ['cfs', 'total_flow', 'quarterly_flow']
            }
        },
        na_rep='Missing'
    )
    .bar(
        subset='cfs',
        color='#c87f0f',
        vmax=agg_flow.cfs.quantile(.95)
    )
    .bar(
        subset='off_goal',
        color='red',
        vmin=-1,
        align='mid'
    )
    .highlight_null(color='#feff7f')
    .highlight_max(color='lightgreen', axis='index')
)

In [ ]:
(
    agg_flow  # HIDECELL
    .fillna(0)  # .bar doesn't work with NaNs
    .reset_index()
    .style

    # after this we are not working with a dataframe but a "Styler" object
    .format(
        {
            'cost': '${:,.0f}',
            'datetime': '{:%Y/%m}/01',
            'percent_quarterly_flow': '{:.1%}',
            'off_goal': '{:.1%}',

            **{
                col: '{:.1f}'
                for col in ['cfs', 'total_flow', 'quarterly_flow']
            }
        },
        na_rep='Missing'
    )

    .bar(
        subset='cfs',
        color='#c87fef',
        vmax=agg_flow.cfs.quantile(.95)
    )

    .bar(
        subset='off_goal',
        color=['red', 'green'],
        align='mid'
    )

    .highlight_null(color='#fef70c')

    .highlight_max(
        color='lightgreen',
        axis='index'
    )

    .background_gradient(
        subset='cost',
        axis='index',
        cmap='Reds',
        vmin=1_000,
        vmax=25_000
    )

    .set_caption('Dirty Devil River Flow')

    .set_properties(
        **{'background-color': '#999'},
        subset='datetime'
    )

    .map(
        lambda val:
            (
                f'color: grey; opacity: 80%; '
                f'background-color: {"#4589ae" if val > 50 else "#a05cbc"}'
            ),
        subset='cfs'
    )

    .set_table_styles(
        [{
            'selector': 'th:hover',
            'props': 'background-color: pink; font-size:14pt;'
        }]
    )
)

In [ ]:
(
    agg_flow  # HIDECELL
    .fillna(0)  # .bar doesn't work with NaNs
    .reset_index()
    .style

    # after this we are not working with a dataframe but a "Styler" object
    .format(
        {
            'cost': '${:,.0f}',
            'datetime': '{:%Y/%m}/01',
            'percent_quarterly_flow': '{:.1%}',
            'off_goal': '{:.1%}',

            **{
                col: '{:.1f}'
                for col in ['cfs', 'total_flow', 'quarterly_flow']
            }
        },
        na_rep='Missing'
    )

    .bar(
        subset='cfs',
        color='#c87fef',
        vmax=agg_flow.cfs.quantile(.95)
    )

    .bar(
        subset='off_goal',
        color=['red', 'green'],
        align='mid'
    )

    .highlight_null(color='#fef70c')

    .highlight_max(
        color='lightgreen',
        axis='index'
    )

    .background_gradient(
        subset='cost',
        axis='index',
        cmap='Reds',
        vmin=1_000,
        vmax=25_000
    )

    .set_caption('Dirty Devil River Flow')

    .set_properties(
        **{'background-color': '#999'},
        subset='datetime'
    )

    .map(
        lambda val:
            (
                f'color: grey; opacity: 80%; '
                f'background-color: {"#4589ae" if val > 50 else "#a05cbc"}'
            ),
        subset='cfs'
    )

    .set_table_styles(
        [{
            'selector': 'th:hover',
            'props': 'background-color: pink; font-size:14pt;'
        }]
    )

    .set_sticky(axis='columns')

    .hide(axis='index')
)


### Final Styling Code

In [ ]:
(
    agg_flow  # HIDECELL
    .fillna(0)  # .bar doesn't work with NaNs
    .reset_index()
    .style

    # after this we are not working with a dataframe but a "Styler" object
    .format(
        {
            'cost': '${:,.0f}',
            'datetime': '{:%Y/%m}/01',
            'percent_quarterly_flow': '{:.1%}',
            'off_goal': '{:.1%}',

            **{
                col: '{:.1f}'
                for col in ['cfs', 'total_flow', 'quarterly_flow']
            }
        },
        na_rep='Missing'
    )

    .bar(
        subset='cfs',
        color='#c87fef',
        vmax=agg_flow.cfs.quantile(.95)
    )

    .bar(
        subset='off_goal',
        color=['red', 'green'],
        align='mid'
    )

    .highlight_null(color='#fef70c')

    .highlight_max(
        color='lightgreen',
        axis='index'
    )

    .background_gradient(
        subset='cost',
        axis='index',
        cmap='Reds',
        vmin=1_000,
        vmax=25_000
    )

    .set_caption('Dirty Devil River Flow')

    .set_properties(
        **{'background-color': '#999'},
        subset='datetime'
    )

    .map(
        lambda val:
            (
                f'color: grey; opacity: 80%; '
                f'background-color: {"#4589ae" if val > 50 else "#a05cbc"}'
            ),
        subset='cfs'
    )

    .set_table_styles(
        [{
            'selector': 'th:hover',
            'props': 'background-color: pink; font-size:14pt;'
        }]
    )

    .set_sticky(axis='columns')

    .hide(axis='index')
)

In [ ]:
pd.options.display.max_rows

In [ ]:
with pd.option_context(
    'display.max_rows', 4,
    'display.max_columns', 2
):
    print(agg_flow)

In [ ]:
pd.describe_option('display.max_rows')

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=c0244bfe-23de-4967-9a94-d52f41800dd5' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>